In [2]:
'''***********DO NOT RUN, THESE ARE THE TABLES************
USERS
+---------+---------+---------------------+-------+
| user_id |  name   |        email        | region|
+---------+---------+---------------------+-------+
|   1     | Alice   | alice@example.com   | East  |
|   2     | Bob     | bob@example.com     | West  |
|   3     | Charlie | charlie@example.com | East  |
|   4     | David   | david@example.com   | West  |
|   5     | Emma    | emma@example.com    | East  |
+---------+---------+---------------------+-------+



ORDERS
+----------+---------+------------+--------+
| order_id | user_id |  product   | amount |
+----------+---------+------------+--------+
|   101    |    1    | Phone      | 700    |
|   102    |    2    | Console    | 400    |
|   103    |    1    | Headphones | 150    |
|   104    |    3    | Keyboard   | 120    |
|   105    |    2    | Tablet     | 500    |
|   106    |    3    | Monitor    | 250    |
|   107    |    3    | Mouse      | 50     |
|   108    |    1    | Smartwatch | 300    |
+----------+---------+------------+--------+


PRODUCTS
+-----------+---------------+---------+
| product_id| product_name  |  price  |
+-----------+---------------+---------+
|    1      | Phone         | 700     |
|    2      | Console       | 400     |
|    3      | Headphones    | 150     |
|    4      | Keyboard      | 120     |
|    5      | Tablet        | 500     |
|    6      | Monitor       | 250     |
|    7      | Mouse         | 50      |
|    8      | Smartwatch    | 300     |
+-----------+---------------+---------+   '''

'***********DO NOT RUN, THESE ARE THE TABLES************\nUSERS\n+---------+---------+---------------------+-------+\n| user_id |  name   |        email        | region|\n+---------+---------+---------------------+-------+\n|   1     | Alice   | alice@example.com   | East  |\n|   2     | Bob     | bob@example.com     | West  |\n|   3     | Charlie | charlie@example.com | East  |\n|   4     | David   | david@example.com   | West  |\n|   5     | Emma    | emma@example.com    | East  |\n+---------+---------+---------------------+-------+\n\n\n\nORDERS\n+----------+---------+------------+--------+\n| order_id | user_id |  product   | amount |\n+----------+---------+------------+--------+\n|   101    |    1    | Phone      | 700    |\n|   102    |    2    | Console    | 400    |\n|   103    |    1    | Headphones | 150    |\n|   104    |    3    | Keyboard   | 120    |\n|   105    |    2    | Tablet     | 500    |\n|   106    |    3    | Monitor    | 250    |\n|   107    |    3    | Mouse  

In [3]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 6.4 MB/s eta 0:00:00


In [4]:
# redis_crud.py
# Requires: pip install redis

from typing import Optional, Dict, Any, List
import redis

# =========================
# Redis Connection
# =========================
import redis

"""Basic connection example.
"""

import redis

r = redis.Redis(
    host='redis-17263.c89.us-east-1-3.ec2.cloud.redislabs.com',
    port=17263,
    decode_responses=True,
    username="default",
    password="livFTRPNtHYZXrLwAGdX2ulAo3MGOhzN",
)

success = r.set('foo', 'bar')
# True

result = r.get('foo')
print(result)
# >>> bar



# Test connection
try:
    r.ping()
    print("✅ Connected to Redis!")
except redis.ConnectionError as e:
    print("❌ Redis connection failed:", e)


# -----------------
# Helper utilities
# -----------------
def _exists(r: redis.Redis, key: str) -> bool:
    return bool(r.exists(key))


def _to_int(x: Any) -> Optional[int]:
    try:
        return int(x)
    except Exception:
        return None


# =========== USER CRUD ===========
def add_user(user_id: int, name: str, email: str, region: str = "East") -> bool:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "email": email, "region": region})
    pipe.sadd("users", uid)
    pipe.execute()
    return True


def get_user(user_id: int) -> Optional[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    data = r.hgetall(key)
    if not data:
        return None
    return {"user_id": uid, **data}


def update_user_name(user_id: int, name: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_user_email(user_id: int, email: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "email", email))


def delete_user(user_id: int) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False

    order_set_key = f"orders_by_user:{uid}"
    order_ids = list(r.smembers(order_set_key))

    pipe = r.pipeline()
    for oid in order_ids:
        pipe.delete(f"order:{uid}:{oid}")
    pipe.delete(order_set_key)
    pipe.srem("users", uid)
    pipe.delete(key)
    pipe.execute()
    return True


def get_all_users() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("users")])
    out: List[Dict[str, Any]] = []
    for uid in ids:
        u = get_user(uid)
        if u:
            out.append(u)
    return out


# =========== ORDER CRUD ===========
def add_order(user_id: int, order_id: int, product: str, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    if uid is None or oid is None:
        raise ValueError("user_id and order_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return False

    key = f"order:{uid}:{oid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"product": product, "amount": str(amount)})
    pipe.sadd(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


def get_orders_by_user(user_id: int) -> List[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return []

    set_key = f"orders_by_user:{uid}"
    ids = sorted([int(x) for x in r.smembers(set_key)])
    out: List[Dict[str, Any]] = []
    for oid in ids:
        key = f"order:{uid}:{oid}"
        data = r.hgetall(key)
        if data:
            out.append(
                {
                    "user_id": uid,
                    "order_id": oid,
                    "product": data.get("product"),
                    "amount": float(data.get("amount", "0") or "0"),
                }
            )
    return out


def update_order_product(user_id: int, order_id: int, product: str) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "product", product))


def update_order_amount(user_id: int, order_id: int, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "amount", str(amount)))


def delete_order(user_id: int, order_id: int) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


# =========== QUERY HELPERS ===========
def get_orders_for_region(region: str) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        if u.get("region") == region:
            out.extend(get_orders_by_user(u["user_id"]))
    return out


def total_sales_for_user(user_id: int) -> float:
    return sum(o["amount"] for o in get_orders_by_user(user_id))


def orders_amount_gt(threshold: float) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        for o in get_orders_by_user(u["user_id"]):
            if o["amount"] > threshold:
                out.append(o)
    return out


# =========== PRODUCT CRUD ===========
def add_product(product_id: int, name: str, price: float) -> bool:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "price": str(price)})
    pipe.sadd("products", pid)
    pipe.execute()
    return True


def get_product(product_id: int) -> Optional[Dict[str, Any]]:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    data = r.hgetall(key)
    if not data:
        return None

    return {
        "product_id": pid,
        "name": data.get("name"),
        "price": float(data.get("price", "0") or "0"),
    }


def get_all_products() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("products")])
    out: List[Dict[str, Any]] = []
    for pid in ids:
        p = get_product(pid)
        if p:
            out.append(p)
    return out


def update_product_name(product_id: int, name: str) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_product_price(product_id: int, price: float) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "price", str(price)))


def delete_product(product_id: int) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem("products", pid)
    pipe.execute()
    return True


bar
✅ Connected to Redis!


In [6]:
# export_redis_to_json.py
import json
import redis

r = redis.Redis(host='redis-17263.c89.us-east-1-3.ec2.cloud.redislabs.com', port=17263, db=0, password="livFTRPNtHYZXrLwAGdX2ulAo3MGOhzN", decode_responses=True)

out = {}
for key in r.scan_iter(match='*', count=1000):
    t = r.type(key)
    if t == 'string':
        out[key] = r.get(key)
    elif t == 'hash':
        out[key] = r.hgetall(key)
    elif t == 'list':
        out[key] = r.lrange(key, 0, -1)
    elif t == 'set':
        out[key] = list(r.smembers(key))
    elif t == 'zset':
        out[key] = r.zrange(key, 0, -1, withscores=True)
    else:
        out[key] = {"_type": t}
# Save
with open('redis_dump.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)


In [7]:
#ALL 15 USERS
add_user(1, "Alice", "alice@example.com", "East"  )
add_user(2, "Bob", "bob@example.com", "West"  )
add_user(3, "Charlie", "charlie@example.com", "East"  )
add_user(4, "David", "david@example.com", "West"  )
add_user(5, "Emma", "emma@example.com", "East")
add_user(6, "Frank", "frank@example.com", "West")
add_user(7, "Grace", "grace@example.com", "East")
add_user(8, "Heidi", "heidi@example.com", "West")
add_user(9, "Ivan", "ivan@example.com", "East")
add_user(10, "Judy", "judy@example.com", "West")
add_user(11, "Kevin", "kevin@example.com", "East")
add_user(12, "Liam", "liam@example.com", "West")
add_user(13, "Mia", "mia@example.com", "East")
add_user(14, "Noah", "noah@example.com", "West")
add_user(15, "Olivia", "olivia@example.com", "East")

False

In [8]:
#ALL 18 PRODUCTS
add_product(1, "Phone", 700)
add_product(2, "Console", 400)
add_product(3, "Headphones", 150)
add_product(4, "Keyboard", 120)
add_product(5, "Tablet", 500)
add_product(6, "Monitor", 250)
add_product(7, "Mouse", 50)
add_product(8, "Smartwatch", 300)
add_product(9, "Laptop", 1000)
add_product(10, "Camera", 600)
add_product(11, "Speaker", 200)
add_product(12, "Printer", 350)
add_product(13, "Webcam", 80)
add_product(14, "Router", 100)
add_product(15, "External Hard Drive", 120)
add_product(16, "Headset", 70)
add_product(17, "Microphone", 90)
add_product(18, "Drone", 800)

False

In [9]:
#ALL 18 ORDERS
add_order(1, 101, "Phone", 700)
add_order(2, 102, "Console", 400)
add_order(1, 103, "Headphones", 150)
add_order(3, 104, "Keyboard", 120)
add_order(2, 105, "Tablet", 500)
add_order(3, 106, "Monitor", 250)
add_order(3, 107, "Mouse", 50)
add_order(4, 108, "Smartwatch", 300)
add_order(4, 109, "Laptop", 1000)
add_order(5, 110, "Camera", 600)
add_order(5, 111, "Speaker", 200)
add_order(6, 112, "Printer", 350)
add_order(6, 113, "Webcam", 80)
add_order(7, 114, "Router", 100)
add_order(7, 115, "External Hard Drive", 120)
add_order(8, 116, "Headset", 70)
add_order(8, 117, "Microphone", 90)
add_order(9, 118, "Drone", 800)




False

In [23]:
# 1. Retrieve user 1 and write down their name and email.
u1 = get_user(1)
print("1) User 1:", u1["name"], u1["email"])

1) User 1: Alice new_email@example.com


In [24]:
# 2. Retrieve all users and write down how many users the system contains.
users = get_all_users()
print("2) Number of users:", len(users))

2) Number of users: 15


In [12]:
# 3. Retrieve all orders belonging to user 1 and list the product names.
orders_u1 = get_orders_by_user(1)
print("3) User 1 order products:", [o["product"] for o in orders_u1])

[{'user_id': 1, 'order_id': 101, 'product': 'Phone', 'amount': 700.0}, {'user_id': 1, 'order_id': 103, 'product': 'Headphones', 'amount': 150.0}, {'user_id': 1, 'order_id': 108, 'product': 'Smartwatch', 'amount': 300.0}]


In [13]:
# 4. Retrieve all products and write down all product names.
products = get_all_products()
print("4) All product names:", [p["name"] for p in products])

[{'product_id': 1, 'name': 'Phone', 'price': 700.0}, {'product_id': 2, 'name': 'Console', 'price': 400.0}, {'product_id': 3, 'name': 'Headphones', 'price': 150.0}, {'product_id': 4, 'name': 'Keyboard', 'price': 120.0}, {'product_id': 5, 'name': 'Tablet', 'price': 500.0}, {'product_id': 6, 'name': 'Monitor', 'price': 250.0}, {'product_id': 7, 'name': 'Mouse', 'price': 50.0}, {'product_id': 8, 'name': 'Smartwatch', 'price': 300.0}, {'product_id': 9, 'name': 'Laptop', 'price': 1000.0}, {'product_id': 10, 'name': 'Camera', 'price': 600.0}, {'product_id': 11, 'name': 'Speaker', 'price': 200.0}, {'product_id': 12, 'name': 'Printer', 'price': 350.0}, {'product_id': 13, 'name': 'Webcam', 'price': 80.0}, {'product_id': 14, 'name': 'Router', 'price': 100.0}, {'product_id': 15, 'name': 'External Hard Drive', 'price': 120.0}, {'product_id': 16, 'name': 'Headset', 'price': 70.0}, {'product_id': 17, 'name': 'Microphone', 'price': 90.0}, {'product_id': 18, 'name': 'Drone', 'price': 800.0}]


In [15]:
# 5. Retrieve product 3 and write down its product name and price.
p3 = get_product(3)
print("5) Product 3:", p3["name"], p3["price"])


{'product_id': 3, 'name': 'Headphones', 'price': 150.0}


In [14]:
# 6. Retrieve the total sales amount for user 1 and record the value.
total_u1 = total_sales_for_user(1)
print("6) Total sales for user 1:", total_u1)

1150.0


In [16]:
# 7. Retrieve all orders whose amount is greater than 500 and list their order IDs.
gt_500 = orders_amount_gt(500)
print("7) Orders with amount > 500 (order_ids):", [o["order_id"] for o in gt_500])

[{'user_id': 1, 'order_id': 101, 'product': 'Phone', 'amount': 700.0}, {'user_id': 4, 'order_id': 109, 'product': 'Laptop', 'amount': 1000.0}, {'user_id': 5, 'order_id': 110, 'product': 'Camera', 'amount': 600.0}, {'user_id': 9, 'order_id': 118, 'product': 'Drone', 'amount': 800.0}]


In [17]:
# 8. Retrieve all orders placed by users in the East region and list all order IDs.
east_orders = get_orders_for_region("East")
print("8) East-region order IDs:", [o["order_id"] for o in east_orders])

[{'user_id': 1, 'order_id': 101, 'product': 'Phone', 'amount': 700.0}, {'user_id': 1, 'order_id': 103, 'product': 'Headphones', 'amount': 150.0}, {'user_id': 1, 'order_id': 108, 'product': 'Smartwatch', 'amount': 300.0}, {'user_id': 3, 'order_id': 104, 'product': 'Keyboard', 'amount': 120.0}, {'user_id': 3, 'order_id': 106, 'product': 'Monitor', 'amount': 250.0}, {'user_id': 3, 'order_id': 107, 'product': 'Mouse', 'amount': 50.0}, {'user_id': 5, 'order_id': 110, 'product': 'Camera', 'amount': 600.0}, {'user_id': 5, 'order_id': 111, 'product': 'Speaker', 'amount': 200.0}, {'user_id': 7, 'order_id': 114, 'product': 'Router', 'amount': 100.0}, {'user_id': 7, 'order_id': 115, 'product': 'External Hard Drive', 'amount': 120.0}, {'user_id': 9, 'order_id': 118, 'product': 'Drone', 'amount': 800.0}]


In [18]:
# 9. Update the email of user 1, then retrieve the user and record the updated email.
update_user_email(1, "alice_new@example.com")
u1_updated = get_user(1)
print("9) Updated email for user 1:", u1_updated["email"])


{'user_id': 1, 'name': 'Alice', 'email': 'new_email@example.com', 'region': 'East'}


In [20]:
#10. Update the name of product 2, then retrieve the product and write down the updated name.



{'product_id': 2, 'name': 'New Console', 'price': 400.0}


In [22]:
#12. Retrieve the orders of any user of your choice, then retrieve the matching product information for each product name and compare their prices.
user_choice = 1
orders_user_choice = get_orders_by_user(user_choice)
print(orders_user_choice)




[{'user_id': 1, 'order_id': 101, 'product': 'Phone', 'amount': 700.0}, {'user_id': 1, 'order_id': 103, 'product': 'Headphones', 'amount': 150.0}, {'user_id': 1, 'order_id': 108, 'product': 'Smartwatch', 'amount': 300.0}]


In [26]:
#13. Retrieve all products and all orders, then determine whether every product name in the orders exists in the product list.
all_products = get_all_products()
all_orders = get_all_orders





NameError: name 'get_all_orders' is not defined